In [1]:
# %pip install kagglehub

In [2]:
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("dansbecker/food-101")

# print("Path to dataset files:", path)

In [ ]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    TrainingArguments,
    Trainer
)
from torchvision.transforms import (
    RandomResizedCrop,
    Compose,
    Normalize,
    ToTensor,
    RandomHorizontalFlip
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# 데이터셋 로드
# Food-101 5개 클래스만 선택
selected_classes = ['apple_pie','baby_back_ribs', 'baklava', 'beef_carpaccio', 'beef_tartare']

path= r'C:\00AI\LLM\20openai\multimodal_food'
dataset = load_dataset(path, split='train[:1000]') # 로컬데이터 사용
# dataset = load_dataset('food101', split='train[:1000]')  # 허깅페이스 데이터셋 사용
print(dataset)


# 선택한 클래스만 필터링
def filter_classes(dataset):
    return dataset['label'] in range(len(selected_classes))

dataset = dataset.filter(filter_classes)
dataset = dataset.train_test_split(test_size=0.2, seed=42)

print(f"훈련데이터 : {len(dataset['train'])}개")
print(f"테스트 데이터 : {len(dataset['test'])}개")
print(f"클래스 : {selected_classes}")

# 이미지 프로세스 로드
checkpoint = 'google/vit-base-patch16-224'
image_processor = AutoImageProcessor.from_pretrained(checkpoint)
print(f"이미지 크기: {image_processor.size}")
print(f"정규화 mean: {image_processor.image_mean}")
print(f"정규화 std: {image_processor.image_std}")

# 데이터 증강 - 파이프라인
normalize = Normalize(
    mean = image_processor.image_mean,
    std = image_processor.image_std
)

size = ( image_processor.size['shortest_edge'] if 'shortest_edge' in image_processor.size 
        else (image_processor.size['height'], image_processor.size['width']))

# 훈련용 
train_transforms = Compose ([
    RandomResizedCrop(size),
    RandomHorizontalFlip(p=0.5),
    ToTensor(),
    normalize
])

# 검증용 데이터 변환 (데이터 증강 없음)
val_transforms = Compose ([
    RandomResizedCrop(size),
    ToTensor(),
    normalize
])

# 변환적용
def process_train(examples):
    examples['pixel_values'] = [
        train_transforms (img.convert('RGB')) for img in examples['image']
    ]
    examples.pop("image", None)
    return examples

# 변환적용
def process_val(examples):
    examples['pixel_values'] = [
        val_transforms (img.convert('RGB')) for img in examples['image']
    ]
    examples.pop("image", None)
    return examples

train_dataset = dataset['train'].with_transform(process_train)
test_dataset = dataset['test'].with_transform(process_val)

print('\n데이터 증강 파이프라인 설정 완료')

print('\n모델 로드 중 ....')

# 라벨 매핑 생성
labels = selected_classes
label2id = {label : i for i, label in enumerate(labels)}
id2label = {i : label for i, label in enumerate(labels)}

model = AutoModelForImageClassification.from_pretrained(
    checkpoint,
    num_labels = len(labels),
    id2label = id2label,
    label2id = label2id,
    ignore_mismatched_sizes = True, # 분류헤더의 크기가 불일치해도 무시하고 분류해
)

# 평가 메트릭 정의
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis = 1)
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='weighted'
    )
    return {
        'accuracy' : accuracy,
        'precision' : precision,
        'recall' : recall,
        'f1' : f1
    }

# 학습 설정
training_args = TrainingArguments(
    output_dir = './13multimodal_food/vit_finetuned_food101',
    remove_unused_columns=False,
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate = 5e-5, #트랜스포머 계열은 5e-5 
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    logging_dir='./vit_finetuned_food101/log',
    save_total_limit=2,
    seed=42
)

# Trainer 생성 및 학습
trainer = Trainer(
    model=model,
    args = training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics = compute_metrics
)

print('\n학습을 시작합니다.')

try :
    result = trainer.train()
    # 최종평가
    eval_results = trainer.evaluate()
    print('\n학습완료')
    print(f"정확도(accuracy): {eval_results['eval_accuracy']:.4f}")
    print(f"정밀도(precision): {eval_results['eval_precision']:.4f}")
    print(f"재현율(recall): {eval_results['eval_recall']:.4f}")
    print(f"f1 점수(f1-score): {eval_results['eval_f1']:.4f}")

    # 모델 저장
    trainer.save_model('./vit_finetuned_food101/final')
    print(f"모델 저장 완료 : ./vit_finetuned_food101/final")

except Exception as e:
    print(f"error:{e}")


Resolving data files:   0%|          | 0/95 [00:00<?, ?it/s]

Dataset({
    features: ['image', 'label'],
    num_rows: 95
})
훈련데이터 : 76개
테스트 데이터 : 19개
클래스 : ['apple_pie', 'baby_back_ribs', 'baklava', 'beef_carpaccio', 'beef_tartare']


Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


이미지 크기: {'height': 224, 'width': 224}
정규화 mean: [0.5, 0.5, 0.5]
정규화 std: [0.5, 0.5, 0.5]

데이터 증강 파이프라인 설정 완료

모델 로드 중 ....


Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([5]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([5, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



학습을 시작합니다.


c:\Users\playdata2\miniconda3\envs\P10\lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.985834,0.736842,0.838596,0.736842,0.729346
2,No log,0.694893,0.842105,0.865789,0.842105,0.840017
3,No log,0.529540,0.894737,0.894737,0.894737,0.894737
4,No log,0.435112,0.947368,0.960526,0.947368,0.947368
5,No log,0.431341,0.894737,0.894737,0.894737,0.894737


c:\Users\playdata2\miniconda3\envs\P10\lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\playdata2\miniconda3\envs\P10\lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\playdata2\miniconda3\envs\P10\lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\playdata2\miniconda3\envs\P10\lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\playdata2\miniconda3\envs\P10\lib\s


학습완료
정확도(accuracy): 0.9474
정밀도(precision): 0.9605
재현율(recall): 0.9474
f1 점수(f1-score): 0.9491
모델 저장 완료 : ./vit_finetuned_food101/final


## fine-tuning 데이터

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("dansbecker/food-101")

print("Path to dataset files:", path)